# Intro
Databricks et plus spécifiquement son environnement de developpement python repose sur Spark (cf. [Documentation PySpark](https://spark.apache.org/docs/latest/api/python/index.html)) qui permet d'interragir et de traiter des gros volumes de données en Python.

Dans ce notebook : 
- Les fonctions principales de PySpark et manipulation de Dataframes



## Les fonctions principales de PySpark et manipulation de Dataframes
- Création de dataframe
- select
- filtres
- tri (sort)
- aggrégations
- join

### Création de Dataframe

In [0]:

# Exemple simple
data = [("Alice", 25), ("Bob", 30), ("Charlie", 35)]
columns = ["name", "age"]

df = spark.createDataFrame(data, columns)

# Afficher le contenu
display(df)


In [0]:

import pandas as pd

# Créer un DataFrame Pandas
pdf = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie"],
    "age": [25, 30, 35]
})

# Convertir en Spark DataFrame
df = spark.createDataFrame(pdf)

display(df)

### Select sur un Dataframe

In [0]:

# Exemple : sélectionner uniquement "name" et "age"
df_selected = df.select("name")

display(df_selected)

In [0]:
df_age = df.select(df.columns[1])
display(df_age)

### Filtre sur un dataframe

In [0]:
df_filtered_age = df.filter(df.age > 30)
display(df_filtered_age)

df_filtered_name = df.filter(df.name == "Alice")
display(df_filtered_name)

In [0]:
# on peut aussi le faire avec des conditions avec un style SQL - LIKE
df_filtered_name_sql = df.filter("name == 'Alice'")
display(df_filtered_name_sql)

### Sorting sur un dataframe

In [0]:
# dans un premier temps il faut import la lib functions de pypspark.sql
from pyspark.sql import functions as F

display(df)
# Tri decroissant sur l'age
df_sorted_age = df.orderBy(F.col("age").desc())
display(df_sorted_age)

# on peut aussi utiliser la fonction sort
df_sorted_age_2 = df.sort(F.col("age").desc())
display(df_sorted_age_2)

In [0]:
# on peut egalement faire des tri sur plusieurs colonnes
df_sorted_age_name = df.sort(F.col("age").desc(), F.col("name").asc())
display(df_sorted_age_name)

### Aggrégations sur un dataframe

In [0]:
# ajoutons de nouvelles lignes dans le dataframe pour que l'exemple soit plus parlant
# df existant avec colonnes: id, name, age
new_rows = [
    {"name": "Alice", "age": 27},
    {"name": "Bob",   "age": 35},
]

df_new = spark.createDataFrame(new_rows)
df_appended = df.unionByName(df_new)   # colonnes alignées par nom


In [0]:
display(df_appended)

# Exemple de base : stats par nom
df_grouped = (
    df_appended.groupBy("name")
      .agg(
          F.count("*").alias("nb_rows"),
          F.avg("age").alias("avg_age")
      )
)

display(df_grouped)


### JOIN dataframes

Types de join utiles

- inner : garde les correspondances
- left : garde tout df1 + correspondances df2
- right : garde tout df2 + correspondances df1
- full : garde tout des deux
- left_semi : garde lignes de df1 qui ont une correspondance dans df2 (sans colonnes df2)
- left_anti : garde lignes de df1 qui n’ont pas de correspondance dans df2

In [0]:
# DataFrame 1
df1 = spark.createDataFrame([
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie")
], ["id", "name"])

# DataFrame 2
df2 = spark.createDataFrame([
    (1, "Paris"),
    (2, "Lyon"),
    (4, "Marseille")
], ["id", "city"])

# Join inner sur la colonne id
df_joined = df1.join(df2, df1.id == df2.id, "inner")

display(df_joined)

# si les colonnes ont le même nom ont peu utiliser using pour simplifier le dataframe final :
df_joined = df1.join(df2, "id", "inner")

display(df_joined)
